# Lesson 4 — RAG cost: chunking, k, and the token tax

**Module 2 · ~12 minutes · API key required for the k-sweep** (chunking is offline)

In most RAG stacks, `k=20` is a default nobody chose. Each extra chunk is billed as input on every query. This lesson shows that the cheapest ingest strategy won the 2026 accuracy benchmarks — and that retrieval recall is not answer quality.

No vector database. We use a tiny in-memory cosine search so the cost mechanics are visible.

### What you will be able to do

1. See how chunk size trades storage/embed cost against tokens dragged into context.
2. Sweep `k` and find the smallest k that still answers correctly — with dollars next to the score.
3. Compare "stuff the whole corpus" vs retrieve-a-few, and decide whether the accuracy gain is worth the multiple.


### How to work through this notebook

Run cells **top to bottom**. Each section tells you what is about to happen *before* you run the code.

| Marker | What it means |
|---|---|
| **About to happen** | What the next cell will do |
| **Watch for** | The number or field that makes the point — pause on it |
| **Why it matters** | The Monday-morning decision this should change |
| **Presenting:** | Live-demo cue. Students: treat this as the takeaway |

A **cost ledger** prints at the end of every notebook that spends money.


In [1]:
import sys
from pathlib import Path

HERE = Path.cwd()
for candidate in (HERE, HERE / "notebooks"):
    if (candidate / "coursekit.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError(
        "coursekit.py not found. Open Jupyter from the course repo root "
        "or from the notebooks/ folder."
    )

import matplotlib.pyplot as plt
import pandas as pd

from coursekit import (
    boot, complete, cost, usd, ntok, token_pieces, forecast, ledger,
    log_call, cache_breakeven, PRICES, MODELS, show,
)

cfg = boot()
print(f"\nLive provider: {cfg.provider}")
print("Need the other vendor? Set LLM_PROVIDER=openai or LLM_PROVIDER=anthropic in .env and re-run this cell.")


  Provider : none (offline)
  Arithmetic cells still run. Live cells will use rehearsal fallbacks.
  Add OPENAI_API_KEY or ANTHROPIC_API_KEY to .env for live calls.
  Rate card: verified 5 Sep 2026 — re-check before presenting.

Live provider: offline
Need the other vendor? Set LLM_PROVIDER=openai or LLM_PROVIDER=anthropic in .env and re-run this cell.


The cell above loads `.env`, chooses **OpenAI or Anthropic** from the key you have, and prints the three model tiers this notebook will call.

**Watch for:** a banner with `Provider`, `floor`, `mid`, `frontier`.
- If it names a vendor, live cells will spend a few cents.
- If it says `offline`, arithmetic still runs. Live cells print a rehearsal fallback instead of crashing — useful on a plane, not a substitute for a real key on caching / routing / eval lessons.

Switch vendor by setting `LLM_PROVIDER=openai` or `LLM_PROVIDER=anthropic` in `.env` and re-running that cell.


## 1. A small corpus

**About to happen.** Five real policy documents plus 25 filler pages about warehouse operations. The filler exists so over-retrieval has something useless to retrieve — which is what `k=20` does in production.

**Watch for:** 30 documents. The question we will ask is only answered in `refunds`.


In [2]:
DOCS = {
    "refunds":   "Refunds under $500 are auto-approved when a shipment is delayed more than 48 hours "
                 "and the customer has fewer than three claims in the trailing twelve months. "
                 "Claims above $500 require supervisor approval and a case note.",
    "tracking":  "Tracking numbers use the format NW-########. Customers can track shipments in the "
                 "portal or by SMS. Tracking data refreshes every 30 minutes from carrier feeds.",
    "delays":    "A shipment is considered delayed when it exceeds the committed delivery window by "
                 "more than 24 hours. Weather exclusions apply during declared severe weather events.",
    "claims":    "Damage claims must be filed within 14 days of delivery with photographic evidence. "
                 "Claims filed after 14 days are rejected automatically unless a supervisor overrides.",
    "escalation":"Escalate to a human agent when the customer requests it, when the claim exceeds "
                 "$500, or when the customer has three or more open cases.",
}
for i in range(25):
    DOCS[f"filler_{i}"] = (
        "General information about warehouse operations, shift scheduling, "
        "forklift certification requirements and seasonal staffing patterns. " * 3
    )
print(len(DOCS), "documents")


30 documents


---
## 2. Chunking — the strategy comparison that surprises people

2026 benchmark synthesis (FloTorch on 50 papers / 905,746 tokens; NVIDIA; Chroma Research):

| Strategy | End-to-end accuracy | Model calls at ingest |
|---|---|---|
| **Recursive character, 512 tok** | **69% — best** | **zero** |
| Fixed-size, 512 tok | 67% | zero |
| Semantic | **54%** (91.9% *recall*) | embedding calls |

The cheapest strategy won. Semantic chunking won **retrieval recall** and lost **the answer** — because it produced tiny 43-token fragments that starved the generator.

Overlap is a tunable, not a mandatory default. A 2026 analysis found it added no measurable benefit in its tested setup while raising indexing cost.

**About to happen.** We split this corpus at 256 / 512 / 1024 tokens and print chunk count vs total tokens.

**Watch for:** bigger chunks → fewer chunks → cheaper to embed and store, but more irrelevant text stuffed into every query. That is the trade. We are not picking a winner on this toy corpus — we are making the trade visible.


In [3]:
import tiktoken, textwrap
from collections import Counter
import math

enc = tiktoken.get_encoding("o200k_base")

def chunk_recursive(text, size=512, overlap_pct=0.15):
    toks = enc.encode(text)
    step = max(1, int(size * (1 - overlap_pct)))
    return [enc.decode(toks[i:i + size]) for i in range(0, len(toks), step)]

for size in [256, 512, 1024]:
    chunks = [c for d in DOCS.values() for c in chunk_recursive(d, size)]
    total = sum(ntok(c) for c in chunks)
    print(f"chunk_size={size:>5}  chunks={len(chunks):>4}  total_tokens={total:>6}  "
          f"avg={total // len(chunks):>4}")
print()
print("Bigger chunks -> fewer chunks -> lower embedding and storage cost,")
print("but more irrelevant text dragged into context on every retrieval. That is the trade.")


chunk_size=  256  chunks=  30  total_tokens=  1464  avg=  48
chunk_size=  512  chunks=  30  total_tokens=  1464  avg=  48
chunk_size= 1024  chunks=  30  total_tokens=  1464  avg=  48

Bigger chunks -> fewer chunks -> lower embedding and storage cost,
but more irrelevant text dragged into context on every retrieval. That is the trade.


### Starting points, not laws

- Factoid queries (names, dates, thresholds): **256–512** tokens
- Analytical queries (comparisons, explanations): **512–1,024** tokens
- Overlap: **10–25%** of chunk size — treat as a tunable, measure it


---
## 3. Retrieval — and what `k` actually costs you

**About to happen.** A bag-of-words cosine search (good enough to show the cost mechanics; swap a real embedding in production). We retrieve the top-3 chunks for: *What is the refund threshold and how many prior claims disqualify a customer?*

**Watch for:** whether `refunds` is in the top-3. If filler pages outrank it, you are about to pay to send noise to the model.


In [4]:
def vec(t):
    return Counter(w.lower().strip(".,?$") for w in t.split() if len(w) > 2)

def cos(a, b):
    common = set(a) & set(b)
    num = sum(a[w] * b[w] for w in common)
    den = math.sqrt(sum(v * v for v in a.values())) * math.sqrt(sum(v * v for v in b.values()))
    return num / den if den else 0.0

CHUNKS = [c for d in DOCS.values() for c in chunk_recursive(d, 512)]
CVECS  = [vec(c) for c in CHUNKS]

def retrieve(query, k):
    qv = vec(query)
    scored = sorted(((cos(qv, cv), c) for cv, c in zip(CVECS, CHUNKS)),
                    reverse=True, key=lambda x: x[0])
    return [c for _, c in scored[:k]]

QUESTION = "What is the refund threshold and how many prior claims disqualify a customer?"
print("top-3 retrieved:\n")
for c in retrieve(QUESTION, 3):
    print(" -", textwrap.shorten(c, 110))


top-3 retrieved:

 - Refunds under $500 are auto-approved when a shipment is delayed more than 48 hours and the customer has [...]
 - Escalate to a human agent when the customer requests it, when the claim exceeds $500, or when the [...]
 - Tracking numbers use the format NW-########. Customers can track shipments in the portal or by SMS. [...]


---
## 4. The k sweep — the money slide of this notebook

**About to happen.** The same question at `k = 20, 10, 5, 3`. Each run sends those chunks to the model. We score **grounded** if the answer contains both `$500` and `three`/`3`.

**Watch for:** the smallest `k` that is still `GROUNDED`, and the `%` saving versus k=20. On a well-chunked corpus this is often k=3 or k=5 and an 80%+ input cut.

**Why it matters.** Default k=20 is the bill. Tune k with an eval, not folklore. Retrieve broadly if you need to, then **rerank down** before the generator sees the context.

> **Presenting:** print cost and groundedness together. Never present the 87% token cut without the score.


In [5]:
results = []
for k in [20, 10, 5, 3]:
    ctx = "\n\n".join(retrieve(QUESTION, k))
    r = complete(
        f"CONTEXT:\n{ctx}\n\nQUESTION: {QUESTION}",
        system="Answer only from the provided context. If it is not there, say so.",
        model=MODELS.mid,
        max_tokens=200,
        label=f"k={k}",
    )
    grounded = ("500" in r.text) and ("three" in r.text.lower() or "3" in r.text)
    results.append(dict(k=k, input_tokens=r.fresh_input + r.cache_read,
                        cost=r.usd, grounded=grounded, note="GROUNDED" if grounded else "incomplete"))
    print(f"   k={k}  grounded={grounded}  {r.text[:80]!r}...")

df = pd.DataFrame(results)
df["monthly_at_1M"] = df.cost * 1_000_000
show(df.style.format({"cost": "${:,.6f}", "monthly_at_1M": "${:,.0f}"}))

best = df[df.grounded].iloc[-1] if df.grounded.any() else df.iloc[-1]
worst = df.iloc[0]
print(f"\nSmallest k that still answered correctly: k={int(best.k)}")
print(f"Saving vs k={int(worst.k)}: {1 - best.cost / worst.cost:.0%}  "
      f"({usd(worst.monthly_at_1M)}/mo -> {usd(best.monthly_at_1M)}/mo at 1M queries)")
print("Default k=20 is the bill. Tune k with an eval, not a folklore default.")


⚠ No API key — using a rehearsal result.
k=20                                          $0.002808   in=1004    out=80     cw=0       cr=0       offline fallback
   k=20  grounded=False  '[rehearsal fallback — live API call failed; continuing the teaching arc]'...
⚠ No API key — using a rehearsal result.
k=10                                          $0.001768   in=484     out=80     cw=0       cr=0       offline fallback
   k=10  grounded=False  '[rehearsal fallback — live API call failed; continuing the teaching arc]'...
⚠ No API key — using a rehearsal result.
k=5                                           $0.001248   in=224     out=80     cw=0       cr=0       offline fallback
   k=5  grounded=False  '[rehearsal fallback — live API call failed; continuing the teaching arc]'...
⚠ No API key — using a rehearsal result.
k=3                                           $0.001086   in=143     out=80     cw=0       cr=0       offline fallback
   k=3  grounded=False  '[rehearsal fallback — live 

,k,input_tokens,cost,grounded,note,monthly_at_1M
0,20,1004,$0.002808,False,incomplete,"$2,808"
1,10,484,$0.001768,False,incomplete,"$1,768"
2,5,224,$0.001248,False,incomplete,"$1,248"
3,3,143,$0.001086,False,incomplete,"$1,086"



Smallest k that still answered correctly: k=3
Saving vs k=20: 61%  ($2,808.00/mo -> $1,086.00/mo at 1M queries)
Default k=20 is the bill. Tune k with an eval, not a folklore default.


---
## 5. Long context vs. RAG — the token tax

"The Token Tax of Epistemic Accuracy" (arXiv:2606.20898) found long-context prompting scored **73.1%** vs **65.4%** for semantic RAG — at **26× the per-query token cost**. Long context can be more accurate. It is also a multiple.

**About to happen.** We stuff the entire corpus into one prompt and compare cost to RAG at k=3.

**Watch for:** `TOKEN TAX` — the multiple. On this toy corpus it will not be 26× (the corpus is small). The *shape* is the point: you are buying accuracy with tokens. That is a documented decision, not a default.

**Why it matters.** The question is never "which is better?" It is "what does an error cost us, and is that worth this multiple?"


In [6]:
EVERYTHING = "\n\n".join(DOCS.values())
r = complete(
    f"CONTEXT:\n{EVERYTHING}\n\nQUESTION: {QUESTION}",
    system="Answer only from the provided context.",
    model=MODELS.mid,
    max_tokens=200,
    label="long context (stuff everything)",
)
c_long = r.usd
c_rag = df[df.k == 3].cost.iloc[0]
print(f"\nlong context : {usd(c_long)} per query")
print(f"RAG (k=3)    : {usd(c_rag)} per query")
print(f"TOKEN TAX    : {c_long / c_rag:.1f}x")
print()
print('The question is not "which is better" but')
print('"what does an error cost us, and is that worth this multiple?"')


⚠ No API key — using a rehearsal result.
long context (stuff everything)               $0.003782   in=1491    out=80     cw=0       cr=0       offline fallback

long context : $0.003782 per query
RAG (k=3)    : $0.001086 per query
TOKEN TAX    : 3.5x

The question is not "which is better" but
"what does an error cost us, and is that worth this multiple?"


In [7]:
ledger()



TOTAL SPENT IN THIS NOTEBOOK: $0.0107


,label,model,input,output,cache_write,cache_read,usd,note
0,k=20,claude-sonnet-5,1004,80,0,0,0.002808,offline fallback
1,k=10,claude-sonnet-5,484,80,0,0,0.001768,offline fallback
2,k=5,claude-sonnet-5,224,80,0,0,0.001248,offline fallback
3,k=3,claude-sonnet-5,143,80,0,0,0.001086,offline fallback
4,long context (stuff everything),claude-sonnet-5,1491,80,0,0,0.003782,offline fallback


---
## Takeaways

- Start with **recursive splitting, 512 tokens, 10–25% overlap**. It beat everything more expensive.
- **Tune `k` with an eval, not a default.** It is the largest single dial in the pipeline.
- Rerank a wide candidate set down to a narrow final set — retrieve broadly, send narrowly.
- Long context vs RAG is an accuracy–cost frontier decision. Document where each use case sits.

**Try on Monday:** log `k` and input tokens per RAG query for a week. Re-run a 20-question eval at k=20, 10, 5, 3. Ship the smallest k that holds the score.
